# Part 2: Time series analysis

Owner: _(name)_

Questions this notebook answers, in order:

1. Is there a trend? (price: yes; returns: no)
2. Is there seasonality? (weekday and month effects, probably none worth modelling)
3. Is the series stationary? Plots first, then the ADF and KPSS tests, on the price and on the returns.
4. What do the ACF and PACF say about how far back the past matters?
5. The decision: which series the models forecast, and the exact split dates for `src/config.py`.

### Setup (same cell in every notebook)

Runs in Colab and in Cursor. In Colab it clones the repository first; for a private repository
paste a GitHub token when asked (Settings > Developer settings > Fine-grained tokens, read access
to this repository is enough).

In [ ]:
import sys, pathlib

REPO_URL = "https://github.com/OWNER/sp500-forecasting.git"   # <- set once after the repo exists

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not pathlib.Path("sp500-forecasting").exists() and not pathlib.Path("src/config.py").exists():
    from getpass import getpass
    token = getpass("GitHub token (leave empty if the repo is public): ")
    url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
    import subprocess
    subprocess.run(["git", "clone", "--quiet", url], check=True)
if IN_COLAB and pathlib.Path("sp500-forecasting").exists():
    import os
    os.chdir("sp500-forecasting")

root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src import config, common

df = common.load_prices()
print(root)
print(df.index.min().date(), "to", df.index.max().date(), "|", len(df), "trading days")
df.tail()

## 1. Trend: the price level against the daily return

In [ ]:
r = df["log_return"].dropna()
fig, ax = plt.subplots(2, 1, figsize=(11, 6))
df["Close"].plot(ax=ax[0], title="Close: a long-term direction (trend)")
df["Close"].rolling(250).mean().plot(ax=ax[0], label="250-day rolling mean"); ax[0].legend()
r.plot(ax=ax[1], title="Daily log return: wobbles around zero, no trend")
r.rolling(250).mean().plot(ax=ax[1], label="250-day rolling mean"); ax[1].legend()
plt.tight_layout()

## 2. Does the spread change over time? (rolling standard deviation)

In [ ]:
r.rolling(60).std().plot(figsize=(11, 3), title="60-day rolling standard deviation of returns (volatility clusters)")

## 3. Seasonality

A fixed, known period would be a weekday pattern (period 5) or a month pattern (period 12).
Compare the averages; with returns this close to zero, a visible bar is usually noise.

In [ ]:
by_weekday = r.groupby(r.index.dayofweek).agg(["mean", "std", "count"])
by_weekday.index = ["Mon", "Tue", "Wed", "Thu", "Fri"]
by_month = r.groupby(r.index.month).agg(["mean", "std", "count"])
display(by_weekday.round(5)); display(by_month.round(5))

## 4. Stationarity tests

Two tests with opposite starting assumptions, so read them together:

- **ADF** (augmented Dickey-Fuller) starts from "not stationary". A small p-value (under 0.05) is evidence **for** stationarity.
- **KPSS** starts from "stationary". A small p-value is evidence **against** stationarity.

Stationary series: ADF small, KPSS large. Non-stationary series: ADF large, KPSS small.

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss
import warnings

def stationarity_report(series, name):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")          # KPSS warns when the p-value hits the table limits
        adf_stat, adf_p, *_ = adfuller(series.dropna(), autolag="AIC")
        kpss_stat, kpss_p, *_ = kpss(series.dropna(), regression="c", nlags="auto")
    verdict = "stationary" if (adf_p < 0.05 and kpss_p >= 0.05) else "NOT stationary" if (adf_p >= 0.05 and kpss_p < 0.05) else "mixed, look at the plots"
    print(f"{name:12s} ADF p = {adf_p:.4f}   KPSS p = {kpss_p:.4f}   -> {verdict}")

stationarity_report(df["Close"], "Close")
stationarity_report(r, "log return")

## 5. ACF and PACF: how far back does the past matter?

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
fig, ax = plt.subplots(2, 2, figsize=(12, 7))
plot_acf(df["Close"].dropna(), lags=40, ax=ax[0, 0], title="ACF of Close (decays slowly: not stationary)")
plot_pacf(df["Close"].dropna(), lags=40, ax=ax[0, 1], title="PACF of Close")
plot_acf(r, lags=40, ax=ax[1, 0], title="ACF of log return (inside the limits: close to white noise)")
plot_pacf(r, lags=40, ax=ax[1, 1], title="PACF of log return")
plt.tight_layout()

## 6. Decision

Write the verdict in plain words, for example:

- The price level is not stationary (ADF p large, KPSS p small, ACF decays slowly). One difference
  of the log price, the daily log return, is stationary. So d = 1 for ARIMA on the price, or
  equivalently d = 0 on the return.
- The ACF and PACF of the return show no bar clearly beyond the limits at low lags, so the series
  is close to a random walk, and every model should be compared with that baseline.
- Split dates for `src/config.py`: TRAIN_END = ..., VAL_END = ... (reason: ...).